In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from natsort import natsorted
from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from plot_helpers import plot_jsd_model_prediction_relation

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

## Overview

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "fluid_tank",
    model_class=NeuralEulerODE,
    verbose=False,
    expecting_sub_folders=False,
)

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "pendulum",
    model_class=NeuralEulerODEPendulum,
    verbose=False,
    expecting_sub_folders=False,
)

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "cart_pole",
    model_class=NeuralEulerODECartpole,
    verbose=False,
    expecting_sub_folders=False,
)

## Rollout plots:

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law
from dmpe.evaluation.model_evaluation import RolloutComparison
from plot_helpers import plot_model_rollouts

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "fluid_tank") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[2],
    model_class=NeuralEulerODE,
)

env, penalty_function, featurize, _ = setup_fluid_tank_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    model=result.median_model,
    batch_size=10,
    sequence_length=100,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "pendulum") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[2],
    model_class=NeuralEulerODEPendulum,
)

env, penalty_function, featurize, _ = setup_pendulum_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    model=result.median_model,
    batch_size=10,
    sequence_length=100,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "cart_pole") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[2],
    model_class=NeuralEulerODECartpole,
)

env, penalty_function, featurize, _ = setup_cart_pole_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    model=result.median_model,
    batch_size=10,
    sequence_length=100,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

## Rollout comparison:

In [ ]:
from plot_helpers import plot_jsd_model_rollout_relation
from dmpe.related_work.random_walk import random_walk_control_law

In [ ]:
env, penalty_function, featurize, _ = setup_fluid_tank_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "fluid_tank",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODE,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
env, penalty_function, featurize, _ = setup_pendulum_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "pendulum",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODEPendulum,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "cart_pole",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODECartpole,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

## Indepth:

### Fluid tank:

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "fluid_tank" / "1000_len" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

In [ ]:
env, _, featurize, _ = setup_fluid_tank_env()
wrapped_env = EnvWrapper(env, featurize=lambda x: x)

points_per_dim = 100

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=1,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[2],
    model_class=NeuralEulerODE,
)
print(result.exp_id)

result.visualize_training()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[2], # 2
    model_class=NeuralEulerODE,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=wrapped_env.featurize),
    model_evaluator,
    labels=["h", "q_in"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=1, c="crimson")

plt.show()

In [ ]:
wrapped_model = NodeModelWrapper(result.median_model, featurize=wrapped_env.featurize)

### Pendulum

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "pendulum") + "/*.eqx")

def featurize_theta(obs):
    """The angle itself is difficult to properly interpret in the loss as angles
    such as 1.99 * pi and 0 are essentially the same. Therefore the angle is
    transformed to sin(phi) and cos(phi) for comparison in the loss."""
    feat_obs = jnp.stack([jnp.sin(obs[..., 0] * jnp.pi), jnp.cos(obs[..., 0] * jnp.pi), obs[..., 1]], axis=-1)
    return feat_obs

In [ ]:
env, _, _ = setup_pendulum_env()
wrapped_env = EnvWrapper(env, featurize=featurize_theta)

points_per_dim = 100

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=2,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[-1], # 2
    model_class=NeuralEulerODEPendulum,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize_theta),
    model_evaluator,
    labels=["theta", "omega", "torque"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

### Cart-pole:

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "cart_pole" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)
print("# of dmpe results:", len(result_paths[:60]))
print("# of goats results:", len(result_paths[60:]))

In [ ]:
def featurize_theta_cart_pole(obs):
    """The angle itself is difficult to properly interpret in the loss as angles
    such as 1.99 * pi and 0 are essentially the same. Therefore the angle is
    transformed to sin(phi) and cos(phi) for comparison in the loss."""
    feat_obs = jnp.stack(
        [obs[..., 0], obs[..., 1], jnp.sin(obs[..., 2] * jnp.pi), jnp.cos(obs[..., 2] * jnp.pi), obs[..., 3]],
        axis=-1,
    )
    return feat_obs

In [ ]:
env, _, _ = setup_cart_pole_env()
wrapped_env = EnvWrapper(env, featurize=featurize_theta_cart_pole)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=4,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[-1], # 2
    model_class=NeuralEulerODECartpole,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize_theta_cart_pole),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[65],
    model_class=NeuralEulerODECartpole,
)
print(result.exp_id)

result.visualize_training()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[65],
    model_class=NeuralEulerODECartpole,
)

plot_sequence(
    result.observations,
    result.actions,
    env.tau,
    env.obs_description,
    env.action_description,
)
plt.show()

result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=wrapped_env.featurize),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

plot_feature_combinations(
    data = jnp.concatenate([result.observations, result.actions], axis=-1),
    labels = ["d", "v", "theta", "omega", "u"]
)
plt.show()

In [ ]:
# result = ModelExpDataResult.from_file(
#     filename=result_paths[3], # 2
#     model_class=NeuralEulerODECartpole,
# )

# plot_sequence(
#     result.observations,
#     result.actions,
#     env.tau,
#     env.obs_description,
#     env.action_description,
# )
# plt.show()

# result.visualize_model_prediction_performance(
#     NodeModelWrapper(result.median_model),
#     model_evaluator,
#     labels=["d", "v", "theta", "omega", "u"],
# )

# plot_feature_combinations(
#     data = jnp.concatenate([result.observations, result.actions], axis=-1),
#     labels = ["d", "v", "theta", "omega", "u"]
# )
# plt.show()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[65], # 2
    model_class=NeuralEulerODECartpole,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize_theta_cart_pole),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[2], # 2
    model_class=NeuralEulerODECartpole,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize_theta_cart_pole),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

In [ ]:
fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize_theta_cart_pole),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

- It seems that the models do not struggle to extrapolate to high velocities but they do struggle to extrapolate to higher deflections. That seems odd
- **TODO**: Instead of reducing the 3.2M sample points grid. Compute each slice independently with higher resolution

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[2], # 2
    model_class=NeuralEulerODECartpole,
)
wrapped_model = NodeModelWrapper(result.median_model, featurize=featurize_theta_cart_pole)
labels=["d", "v", "theta", "omega", "u"]
points_per_dim = 100

wrapped_env

In [ ]:
from dmpe.utils.density_estimation import build_grid_2d

In [ ]:
base_grid = build_grid_2d(-1, 1, points_per_dim)
base_grid.shape

In [ ]:
n_features = model_evaluator.obs_dim + model_evaluator.act_dim
obs_dim = model_evaluator.obs_dim
act_dim = model_evaluator.act_dim

fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(9, 9), sharex=True, sharey=True)

base_grid = build_grid_2d(-1, 1, points_per_dim)

for i in range(n_features):
    for j in range(n_features):


        axs[j, i].grid(True)
        axs[j, i].set_xlim(-1.1, 1.1)
        axs[j, i].set_ylim(-1.1, 1.1)

        if i == j:
            continue

        image = jnp.zeros((points_per_dim, points_per_dim))

        for a in tqdm(jnp.linspace(-1, 1, 10)):
            for b in jnp.linspace(-1, 1, 10):
                for c in jnp.linspace(-1, 1, 10):                    
                    grid = jnp.concatenate([base_grid, jnp.ones((base_grid.shape[0], 3)) * jnp.array([a,b,c])[None]], axis=-1)
                    
                    pred = jax.vmap(wrapped_model.step, in_axes=(0, 0, None))(
                        grid[:, : obs_dim], grid[:, obs_dim :], env.tau
                    )
                    pred_gt = jax.vmap(wrapped_env.step, in_axes=(0, 0, None))(
                        grid[:, : obs_dim], grid[:, obs_dim :], env.tau
                    )

                    extra_error = jnp.linalg.norm(jnp.squeeze(pred - pred_gt), axis=-1)
            
                    image = image + extra_error.reshape((points_per_dim, points_per_dim))
                    
        
        axs[j, i].imshow(image, origin="lower", extent=[-1, 1, -1, 1])
        axs[j, 0].set_ylabel(labels[j])

        break
        axs[-1, i].set_xlabel(labels[i])
    break

fig.tight_layout()
data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

In [ ]:
points_per_dim = [100, 100, 10, 10, 10]
dim = len(points_per_dim)

xs = [jnp.linspace(-1, 1, n) for n in points_per_dim]

z_g = jnp.meshgrid(*xs)
z_g = jnp.stack([_x for _x in z_g], axis=-1)
z_g = z_g.reshape(-1, dim)

z_g.shape

In [ ]:
n_features = model_evaluator.obs_dim + model_evaluator.act_dim
obs_dim = model_evaluator.obs_dim
act_dim = model_evaluator.act_dim

fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, n_features, 1).tolist()

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].grid(True)
        axs[j, i].set_xlim(-1.1, 1.1)
        axs[j, i].set_ylim(-1.1, 1.1)

        if i == j:
            continue

        points_per_dim = [10] * n_features
        points_per_dim[i] *= 10
        points_per_dim[j] *= 10

        xs = [jnp.linspace(-1, 1, n) for n in points_per_dim]
        z_g = jnp.meshgrid(*xs)
        z_g = jnp.stack([_x for _x in z_g], axis=-1)
        grid = z_g.reshape(-1, n_features)

        pred = jax.vmap(wrapped_model.step, in_axes=(0, 0, None))(
            grid[:, : obs_dim], grid[:, obs_dim :], env.tau
        )
        pred_gt = jax.vmap(wrapped_env.step, in_axes=(0, 0, None))(
            grid[:, : obs_dim], grid[:, obs_dim :], env.tau
        )
        difference_map = pred - pred_gt
        reshaped_difference_map = difference_map.reshape(
            points_per_dim + [-1]
        )
        abs_map = jnp.linalg.norm(reshaped_difference_map, axis=-1)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        image = jnp.sum(abs_map, axis=reduction_indices)

        # image = image.reshape((100, 100))
        image = jnp.transpose(image, (0, 1) if i > j else (1, 0))
        
        axs[j, i].imshow(image, origin="lower", extent=[-1, 1, -1, 1])
        axs[j, 0].set_ylabel(labels[j])

        axs[-1, i].set_xlabel(labels[i])

fig.tight_layout()

# data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
# n_features = data_points.shape[-1]

# for i in range(n_features):
#     for j in range(n_features):
#         axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.05, c="r")
# axs[j, i].grid(True)
# axs[j, i].set_xlim(-1.1, 1.1)
# axs[j, i].set_ylim(-1.1, 1.1)

# plt.show()